# Homework 3: Bootstrap, Hypothesis Testing, Multiple Testing &

Classification Inference

MSE 125 — Spring 2026

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
# multipletests(pvals, alpha, method='fdr_bh' or 'bonferroni') -> (reject, pvals_adj, ...)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'

## How to use this notebook

Write your answers in the cells marked `# Your code here` or *Your
answer here* below each question. Add more cells if you need them
(Insert \> Code cell or Text cell). Run each code cell with Shift+Enter.

## Submission

Submit your completed notebook (.ipynb) to Gradescope by the due date.
Run all cells top-to-bottom before saving so your outputs are included.

## Problem 1: Does the bootstrap actually deliver 95% coverage?

A skeptical statistician asks the question every method developer
eventually faces: *how do we know the bootstrap CI really covers the
true parameter 95% of the time?* You promise to check. Real datasets are
bad places to test coverage — we don’t know the “true” parameter.
Instead, you’ll use a larger, well-measured dataset as a *pretend
population* whose parameter you already know: the Department of Labor’s
public record of every H-1B visa application filed in fiscal year 2024.
You will test whether a bootstrap CI built from a small sample of
applications can recover the true mean annual wage across the full
population.

In [ ]:
# Download and prepare the H-1B LCA disclosure dataset if it is not already available.
# This creates the file expected by the next cell: data/h1b/h1b_filings.csv
import os
import sys
import subprocess
import urllib.request
import pandas as pd

# The setup cell above defines DATA_DIR = 'data'. This fallback makes the cell safer
# if students run it in isolation.
try:
    DATA_DIR
except NameError:
    DATA_DIR = 'data'

H1B_DIR = os.path.join(DATA_DIR, 'h1b')
os.makedirs(H1B_DIR, exist_ok=True)

DATA_URL = 'https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2024_Q4.xlsx'
XLSX_FILE = os.path.join(H1B_DIR, 'LCA_Disclosure_Data_FY2024_Q4.xlsx')
CSV_FILE = os.path.join(H1B_DIR, 'h1b_filings.csv')

# pandas needs openpyxl to read .xlsx files. Install it only if missing.
try:
    import openpyxl  # noqa: F401
except ImportError:
    print('Installing openpyxl so pandas can read the Excel file...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl'])

# Download the raw Excel file if needed.
if not os.path.exists(XLSX_FILE):
    print('Downloading H-1B data from the Department of Labor (~80 MB; this may take a minute)...')
    urllib.request.urlretrieve(DATA_URL, XLSX_FILE)
    print('Download complete.')
else:
    print(f'Using existing Excel file: {XLSX_FILE}')

# Convert the Excel file to the CSV format used in the rest of the homework.
if not os.path.exists(CSV_FILE):
    print('Converting Excel file to CSV...')
    h1b_raw = pd.read_excel(XLSX_FILE, engine='openpyxl')
    h1b_raw.to_csv(CSV_FILE, index=False)
    print(f'Saved CSV to: {CSV_FILE}')
else:
    print(f'Using existing CSV file: {CSV_FILE}')

In [2]:
h1b = pd.read_csv(f'{DATA_DIR}/h1b/h1b_filings.csv', low_memory=False)
annual = h1b[h1b['WAGE_UNIT_OF_PAY'].str.upper() == 'YEAR'].copy()
annual['wage'] = pd.to_numeric(annual['WAGE_RATE_OF_PAY_FROM'], errors='coerce')
annual = annual[annual['wage'].between(20_000, 500_000)].dropna(subset=['wage'])
wage_population = annual['wage'].values
true_mean = wage_population.mean()
print(f"Pretend population: {len(wage_population):,} H-1B annual wages")
print(f"True population mean wage: ${true_mean:,.2f}")

Pretend population: 112,851 H-1B annual wages
True population mean wage: $132,266.08

**(a)** Draw 100 samples of size $n = 50$ from `wage_population`. Sample
**without** replacement from the population (each of the 100 samples is
a different genuine subset). Then, for each of those samples, sample
**with** replacement when building the bootstrap CI. Use 500 bootstrap
replicates to build a 95% percentile CI for the mean. What fraction of
your 100 CIs contain `true_mean`? How close is that to 95%?

(*Note:* with only 100 outer replications, your coverage number has
Monte Carlo error of roughly $\pm 2\%$ — expect some seed-to-seed
jitter. The overall pattern across $n = 50$ and $n = 15$ should hold,
but exact numbers will vary.)

In [3]:
# outer loop: sample without replacement from wage_population
#   (use np.random.choice(..., replace=False))
# inner loop: bootstrap that sample WITH replacement (use np.random.choice(..., replace=True))
# Your code here

*Your answer here.*

**(b)** Repeat (a) with $n = 15$ instead of $n = 50$. Does the empirical
coverage get better, worse, or stay the same? In 2-3 sentences, explain
why small samples of heavy-tailed data (H-1B wages have a long right
tail — a handful of applications above \$300k, most in the five-figure
range) can produce bootstrap CIs that miscover, even though the
bootstrap is “asymptotically correct.”

In [4]:
# Your code here

*Your answer here.*

**(c) The reply to the skeptic.** Write a 2-3 sentence note the
statistician would accept. It must cite your two coverage numbers from
(a) and (b), and name at least one setting where bootstrap percentile
CIs are known to fail (e.g., small $n$, heavy-tailed data, estimating an
extreme quantile, estimating a parameter on the boundary of its domain).

*Your answer here.*

## Problem 2: Are the tigers endangered?

The fictional nation of Sundavia is deciding whether its forest tigers
belong on the endangered species list. By statute, “endangered” means a
population of 100 or fewer adult tigers. The default under the law is to
protect: unless biologists produce statistical evidence that the
population *exceeds* 100, the tigers stay listed.

Your team goes into the forest and captures 20 tigers, tags them, and
releases them. A month later, the team returns and captures 20 more
tigers. Of the second group, **3 carry tags**.

**(a) Build the null distribution.** Under the null hypothesis $H_0$:
the true tiger population is exactly $N = 100$, simulate the number of
tagged tigers you would see in the second capture. Do this 10,000 times
and plot the resulting distribution.

In [5]:
# Your code here

*Your answer here.*

**(b) Why N = 100 is the conservative null.** The statutory definition
of “endangered” is $N \le 100$, so the true null is *composite*: any
population of 100 or fewer. Simulate the null distribution under
$N = 50$ as well and overlay it on your $N = 100$ histogram. Then
answer, in 2-3 sentences total:

1.  How does the null distribution shift as $N$ shrinks from 100 to 50?
    Explain in one sentence using your overlay.

2.  If the observed recapture count of 3 is not small enough to reject
    at $N = 100$, would the p-value under $N = 50$ be even smaller or
    even larger? Use (i) to answer. What does that imply about using
    $N = 100$ as the null even though the true null is composite?

In [6]:
# Your code here

*Your answer here.*

**(c) The p-value.** You observed 3 tagged tigers in the second capture.
Using your 10,000 simulated values from (a), estimate the one-sided
p-value: $\Pr(X \le 3 \mid N = 100)$, where small recapture counts are
evidence that the true population is *larger* than 100 (which is what
you’d need to remove the tigers from the list). At $\alpha = 0.05$, do
you reject $H_0$? In plain English, what does rejecting — or failing to
reject — mean for the listing decision?

In [7]:
# Your code here

*Your answer here.*

**(d) The policy memo.** Return a Python string `memo` (≤ 4 sentences)
addressed to the Sundavian Minister of Environment. It must (i) state
exactly one recommendation — *keep on the list*, *remove from the list*,
or *commission a larger survey* — and (ii) cite your p-value and what it
means in words. The minister is not a statistician; no jargon, no
equations.

In [8]:
memo = """Your memo here."""
print(memo)

Your memo here.

*Your answer here.*

## Problem 3: Does Google pay H-1B workers more than Infosys?

A labor advocate at the Economic Policy Institute is preparing a brief
on H-1B wage stratification. Two of the largest visa sponsors are Google
(which sponsors mostly senior tech roles) and Infosys (an IT consulting
firm that sponsors mostly mid-level contractors). The advocate wants to
know whether the *raw* wage gap between the two employers is real and
how confidently to cite it. You’ll first decide whether the gap is
statistically meaningful, then quantify its size with a confidence
interval the advocate can put in print.

In [9]:
google = annual.loc[annual['EMPLOYER_NAME'].str.contains('Google', case=False, na=False), 'wage'].values
infosys = annual.loc[annual['EMPLOYER_NAME'].str.contains('INFOSYS', case=False, na=False), 'wage'].values
print(f"Google:   n = {len(google)},  mean = ${google.mean():,.0f}")
print(f"Infosys:  n = {len(infosys)}, mean = ${infosys.mean():,.0f}")

Google:   n = 1426,  mean = $190,445
Infosys:  n = 979, mean = $110,472

**(a) Pick a test, defend the choice.** Run *one* defensible test of the
null hypothesis that Google and Infosys H-1B workers earn the same mean
annual wage — either a Welch’s two-sample t-test or a permutation test
(1000 permutations) on the difference in means. Report the test
statistic (or observed difference) and p-value, and state the conclusion
at $\alpha = 0.05$. Then in one sentence, justify your choice: name a
setting (sample size, distributional shape) where the test you *did not*
pick would give a different answer, and explain why that setting does
not apply here.

In [10]:
# Your code here

*Your answer here.*

**(b)** A p-value alone tells the advocate that the gap is real but not
*how big it is*. Bootstrap a 95% percentile CI for the mean wage
difference (Google − Infosys) using 1000 paired resamples (resample each
employer’s wages independently, with replacement, then take the
difference of means). Report the CI.

In [11]:
# Your code here

*Your answer here.*

**(c) The line for the brief.** Write one sentence the advocate could
quote in the brief. It must (i) state the direction and approximate size
of the wage gap in dollars, (ii) cite your bootstrap CI, and (iii) avoid
the word “p-value” entirely (the brief is for a non-statistical
audience).

*Your answer here.*

## Problem 4: Which H-1B employers pay outside the market?

The Department of Labor’s Wage and Hour Division has flagged H-1B wage
compliance as a 2026 enforcement priority. The Division has authority to
investigate employers whose H-1B wages deviate systematically from the
broader market — either suspiciously low (a possible enforcement target:
employers using H-1B workers to undercut prevailing wages and depress
conditions for U.S. workers in the same role) or unusually high (a
candidate “best-practice” benchmark cited in the annual compliance
report distributed to industry groups).

A junior analyst was asked to produce a first-pass list of wage outliers
from the 20 largest visa sponsors. She ran a quick statistical test for
each employer against the national H-1B mean wage and flagged the
employers whose result was significant at $\alpha = 0.05$. Her
supervisor pushes back with two concerns:

1.  *“Twenty tests at $\alpha = 0.05$ — by chance alone, about one will
    look extreme. How do I know your list isn’t half false alarms?”*
2.  *“You’re comparing a consulting firm full of mid-level contractors
    against a research lab full of senior PhD engineers. Of course they
    look different — but that doesn’t mean either is paying ‘outside the
    market’ for the work they actually buy.”*

The supervisor wants both concerns addressed. Whatever list survives is
handed to a **policy advisor** who picks one of three actions: *publish
it* in the annual compliance report (each named employer will appear in
print and may be quoted by the press), *pass it to the audit team* as an
internal triage list (auditors investigate before any name is made
public), or *shelve it* and commission a better study. Each H-1B filing
record includes the employer, the job title, the wage, the work
location, and several occupational classification codes — you may use
any of these in your analysis.

In [12]:
top20_names = annual['EMPLOYER_NAME'].value_counts().head(20).index.tolist()
national_mean = annual['wage'].mean()
print(f"National H-1B mean wage: ${national_mean:,.0f}")
print(f"Top 20 employers by filings (each has at least {annual['EMPLOYER_NAME'].value_counts().iloc[19]} filings).")

National H-1B mean wage: $132,266
Top 20 employers by filings (each has at least 508 filings).

> **A note on this problem.** This problem is intentionally less
> prescriptive than earlier ones. We do not tell you which test to run,
> which correction to apply, or how to subset the data — your future
> supervisors won’t either. They will describe the policy situation and
> the deliverable, and expect *you* to choose the analysis. The four
> sub-parts below give you all the situational context you should need:
> the consequences of a wrong call, the structure of the data, and the
> supervisor’s two specific concerns. Your job is to translate that
> context into an analysis pipeline.
>
> To keep grading tractable, the problem requires specific artifacts and
> a short decision log. The rubric below evaluates the *quality of your
> choices and reasoning*, not whether they match a hidden answer key —
> multiple paths can earn full credit if defended well.
>
> **Required deliverables.**
>
> 1.  **(a)** A baseline list of “outlier” employers, before any
>     corrections.
> 2.  **(b)** The same list after addressing the supervisor’s first
>     concern (multiple testing).
> 3.  **(c)** A revised list after addressing the supervisor’s second
>     concern (job-mix confounding), plus a 4-row decision log.
> 4.  **(d)** A memo for the policy advisor.
>
> **Grading rubric (50 pts).** Baseline test choice is defensible (5) ·
> multiple-testing correction is matched to the *deliverable*, not
> picked at random (10) · confounding adjustment is a real adjustment,
> not a relabel (10) · decision log states *why*, not just *what* (10) ·
> memo gives one clear recommendation grounded in the numbers (15).

**(a) Replicate the analyst’s baseline.** The analyst’s question for
each of the 20 employers in `top20_names` was simple: does this
employer’s average H-1B wage differ from the national H-1B average? Two
facts about the data matter for choosing a test. First, H-1B wages are
right-skewed — most filings are five-figure salaries, with a long tail
of filings above \$300k. Second, employers vary widely in filing volume;
some of the top-20 have only a few hundred filings while others have
tens of thousands.

Pick a defensible test for the analyst’s question, justify the choice in
1-2 sentences (what does it assume, are those assumptions reasonable
here?), and report the resulting list of “outliers” at $\alpha = 0.05$.

In [13]:
# Your code here

*Your answer here.*

**(b) Address the supervisor’s first concern: multiple testing.** What
the policy advisor does with this list changes which type of error is
most expensive:

-   *Publication scenario.* The list goes verbatim into the annual
    compliance report. Every named employer sees itself in print; a
    wrongly-flagged employer can sue, and the DOL’s credibility takes a
    hit if even one name is later retracted. **A single false positive
    is very expensive.**
-   *Triage scenario.* The list goes to the audit team, who investigate
    before any name is made public. False positives cost auditor time
    but cause no public exposure; a false negative means a real bad
    actor walks free for another year. **A handful of false positives in
    a roughly-correct list is acceptable; missing real cases is the
    worst outcome.**

Decide which scenario your correction is aimed at, pick a
multiple-testing correction that matches it, apply it to the (a)
p-values, and report how the list changes. In 2-3 sentences, justify
your choice in terms of the false-positive vs. false-negative cost
trade-off above and state which scenario you are optimizing for.

In [14]:
# Your code here

*Your answer here.*

**(c) Address the supervisor’s second concern: job-mix confounding.**
The supervisor’s worry is concrete. Imagine two employers in
`top20_names`: a research lab that hires only senior PhD ML engineers,
and a consulting firm that hires only associate-level IT contractors.
The lab will look high against the national mean and the consulting firm
will look low — but neither is necessarily paying “outside the market”
for what it actually buys; they’re buying different work at different
price points. The (a) test cannot tell those two stories apart from
genuine wage abuse.

Each filing has a `JOB_TITLE` field that you can use to define
“comparable work,” but be warned: title strings are free-text and
inconsistent. The same role appears as “Software Engineer”, “SOFTWARE
ENGINEER”, “Sr. Software Eng. II”, “Software Developer”, and many other
variants — all stored as separate values. (The dataset also has
`SOC_CODE` / `SOC_TITLE` occupational classification fields if you want
a coarser, cleaner alternative to job titles; either approach is fine if
you justify it.)

Design an analysis that strips out as much of this job-mix confounding
as you can, run it on the same 20 employers, and report which employers’
significance status flips relative to (b). Pick the one employer whose
status changed most dramatically and explain in 1-2 sentences what that
flip tells you about the original analysis.

You will face several judgment calls. Make them, justify them briefly in
the **decision log** below, and note explicitly what your adjustment
does *not* control for (e.g., experience level, geography, industry
sector):

-   How to define “comparable work” — pick a single subset of filings to
    compare within, or stratify and combine across strata
-   How to handle employers whose comparable subset is small — dropping
    them costs you information about that employer entirely; keeping
    them means a noisy, low-power estimate
-   What baseline to compare each employer to, given the slicing choices
    above

**Decision log** — fill in this table (or paste it as markdown):

| Decision                           | What you chose | Why |
|------------------------------------|----------------|-----|
| Test in (a)                        |                |     |
| Multiple-testing correction in (b) |                |     |
| Job-mix adjustment in (c)          |                |     |
| Sample-size threshold for (c)      |                |     |

In [15]:
# Your code here

*Your answer here.*

**(d) The memo to the policy advisor.** The policy advisor will read
your memo and choose among three actions: *publish the list* in the
annual compliance report (names appear in print), *audit the list
internally* (audit team investigates, no public naming yet), or *do not
act on the list* (collect better data first). The advisor is not a
statistician — no equations, no jargon a non-statistician would reach
for a dictionary to read.

Return a Python string `memo` (≤ 5 sentences) that: (i) states how many
employers are still flagged after both concerns are addressed, (ii)
names one specific employer whose status changed once you adjusted for
job mix and what that flip implies, (iii) ends with one concrete
recommendation: *publish the list*, *audit the list internally*, or *do
not act on the list*.

In [16]:
memo = """Your memo here."""
print(memo)

Your memo here.

*Your answer here.*

## Problem 5: Build the shot-quality model — and tell the coach what to do with it

A consulting NBA team has hired you to settle two arguments at once. The
front office wants a quick predictive model — given a shot’s distance
and type, how likely is it to go in? The shooting coach wants a sharper
version of the Moreyball question — for a specific *high-volume
shooter*, do the data say he should keep taking mid-range twos, or trade
them for threes? You’ll build the front-office’s model first (and
quantify how much better it is than a coin flip), then use the
per-player shot record to deliver an actionable recommendation in a text
message the coach can read on the team bus.

In [ ]:
# Download and prepare the NBA shot dataset if it is not already available.
# This creates the file expected below: data/nba/nba_shots_2017_18.csv
import os
import urllib.request
import zipfile
import shutil
import pandas as pd

# The setup cell above defines DATA_DIR = 'data'. This fallback makes the cell safer
# if students run Problem 5 in isolation.
try:
    DATA_DIR
except NameError:
    DATA_DIR = 'data'

NBA_DIR = os.path.join(DATA_DIR, 'nba')
os.makedirs(NBA_DIR, exist_ok=True)

NBA_ZIP_URL = 'https://raw.githubusercontent.com/stanford-mse-125/web/main/homework/nba_shots_2017_18.csv.zip'
NBA_ZIP_FILE = os.path.join(NBA_DIR, 'nba_shots_2017_18.csv.zip')
NBA_CSV_FILE = os.path.join(NBA_DIR, 'nba_shots_2017_18.csv')

if not os.path.exists(NBA_CSV_FILE):
    if not os.path.exists(NBA_ZIP_FILE):
        print('Downloading NBA shot dataset from GitHub...')
        urllib.request.urlretrieve(NBA_ZIP_URL, NBA_ZIP_FILE)
        print('Download complete.')

    print('Unzipping NBA shot dataset...')
    with zipfile.ZipFile(NBA_ZIP_FILE, 'r') as zf:
        csv_files = [name for name in zf.namelist() if name.endswith('.csv')]
        if len(csv_files) == 0:
            raise FileNotFoundError('No CSV file found inside the downloaded zip file.')
        with zf.open(csv_files[0]) as source, open(NBA_CSV_FILE, 'wb') as target:
            shutil.copyfileobj(source, target)
    print(f'Saved CSV to {NBA_CSV_FILE}')
else:
    print(f'Using existing file: {NBA_CSV_FILE}')

shots = pd.read_csv(f'{DATA_DIR}/nba/nba_shots_2017_18.csv').dropna(
    subset=['SHOT_DISTANCE', 'SHOT_MADE_FLAG']
)
shots['is_3pt'] = (shots['SHOT_TYPE'] == '3PT Field Goal').astype(int)
print(f"{len(shots):,} shots in the 2017-18 season")
print(f"League FG%: {shots['SHOT_MADE_FLAG'].mean():.3f}")


**(a) Fit the front-office model.** Split the shots into 80% train / 20%
test (`train_test_split` with `random_state=42`). Fit a logistic
regression predicting `SHOT_MADE_FLAG` from the two features
`SHOT_DISTANCE` and `is_3pt`. Report the test-set AUC.

In [18]:
# Your code here

*Your answer here.*

**(b) Bootstrap CI for AUC.** Resample test rows with replacement (1000
bootstrap samples), recompute AUC each time, and report a 95% percentile
CI for the test AUC. Does the CI exclude 0.5? In one sentence, say what
the CI tells the front office that a single AUC number does not.

In [19]:
# Your code here

*Your answer here.*

**(c) The coach’s question — per-player EPA test.** Identify the 20
highest-volume shooters in the season, then keep only those with at
least 50 attempts in *both* the mid-range two-point zone (2-pointer with
`SHOT_DISTANCE` $> 5$) and the 3-point zone. For each qualifying player,
test $H_0$: the player’s true mid-range expected value $\ge$ their true
3-point expected value. A small p-value means the data favors switching
to 3s. Use a one-sided z-test on the EPA difference
$3 \widehat p_{3\text{pt}} - 2 \widehat p_{\text{mid}}$. With 50+
attempts in each zone, the CLT makes this difference approximately
normal, so a z-test is appropriate (and faster to run 20 times than a
bootstrap). (*Hint:* the standard error of
$a \widehat p_1 - b \widehat p_2$ with independent samples is
$\sqrt{a^2 \widehat p_1(1-\widehat p_1)/n_1 + b^2 \widehat p_2(1-\widehat p_2)/n_2}$.)
Summarize the 20 p-values in a table.

In [20]:
# Your code here

*Your answer here.*

**(d)** Apply the Benjamini-Hochberg correction at FDR = 0.05 to the 20
player tests. How many players can you confidently say are better off
shooting 3s than mid-range? Distinguish players whose corrected p-value
is *near* 0.05 (the data is genuinely ambiguous) from players whose
p-value is *far above* 0.05 (the data clearly does not support a
switch).

In [21]:
# Your code here

*Your answer here.*

**(e) The text to the coach.** A coach argues: “My star player shoots
48% from mid-range. He’s the exception — he should keep taking those
shots.” Compare the expected points per attempt for a 48% mid-range
shooter vs. the league-average 3-point shooter (use the league 3pt FG%
you can compute from the data). Deliver your reply as a two-sentence
text message the coach will read on the team bus: sentence 1 states your
recommendation, sentence 2 makes the opportunity-cost argument in
numbers he will not have to look up.

In [22]:
# Your code here

*Your answer here.*